# Sprint 3 — Geração de Linguagem Natural para Alertas e Estado Operacional

**Challenge FIAP / Forzy — PLN para digital-twin de motores elétricos industriais**

Este notebook cobre:
1. Geração automática de resumos textuais de alertas (templates leve/moderado/crítico)
2. Classificação textual de eventos (zero-shot com LLM/transformers)
3. Relatório operacional em linguagem natural com rastreabilidade
4. Avaliação: ROUGE (resumos) e F1-score por categoria (classificação)

In [ ]:
# Instalação de dependências (rodar no Colab)
# !pip -q install pandas rouge-score scikit-learn transformers torch --upgrade

In [ ]:
import pandas as pd
import json
from pathlib import Path
from io import StringIO

# Dados embutidos diretamente no notebook para que ele rode de forma autossuficiente
# em qualquer ambiente (ex.: upload avulso no Google Colab, sem a pasta data/ ao lado).
# Se a pasta ../data existir (execução local a partir do repositório), ela tem prioridade.
DATA_DIR = Path('../data')

ALERTS_RAW_CSV = """alert_id,motor_id,timestamp,sensor,parametro,valor_medido,baseline,desvio,unidade,nivel,evento_tipo
A001,MT-042,2026-08-18 08:15:00,temp_enrolamento,temperatura,78,60,18,°C,moderado,anomalia_eletrica
A002,MT-017,2026-08-18 09:40:00,vibracao_radial,vibracao,2.1,1.0,1.1,mm/s,critico,anomalia_mecanica
A003,MT-005,2026-08-18 10:05:00,corrente_fase,corrente,12.5,11.0,1.5,A,leve,operacao_normal
A004,MT-042,2026-08-18 12:30:00,temp_enrolamento,temperatura,64,60,4,°C,leve,operacao_normal
A005,MT-029,2026-08-18 14:00:00,vibracao_axial,vibracao,3.4,1.2,2.2,mm/s,critico,anomalia_mecanica
A006,MT-011,2026-08-18 15:20:00,isolamento,resistencia,0.4,5.0,-4.6,MΩ,critico,anomalia_eletrica
A007,MT-017,2026-08-18 16:50:00,rolamento_temp,temperatura,55,45,10,°C,moderado,manutencao_preventiva
A008,MT-005,2026-08-19 07:10:00,corrente_fase,corrente,11.2,11.0,0.2,A,leve,operacao_normal
A009,MT-029,2026-08-19 08:35:00,vibracao_axial,vibracao,1.5,1.2,0.3,mm/s,leve,manutencao_preventiva
A010,MT-011,2026-08-19 09:55:00,isolamento,resistencia,3.2,5.0,-1.8,MΩ,moderado,anomalia_eletrica
A011,MT-042,2026-08-19 11:15:00,temp_enrolamento,temperatura,92,60,32,°C,critico,anomalia_eletrica
A012,MT-017,2026-08-19 13:00:00,vibracao_radial,vibracao,1.3,1.0,0.3,mm/s,leve,manutencao_preventiva
A013,MT-005,2026-08-19 14:25:00,rolamento_temp,temperatura,70,45,25,°C,critico,anomalia_mecanica
A014,MT-029,2026-08-19 15:45:00,corrente_fase,corrente,15.8,11.0,4.8,A,moderado,anomalia_eletrica
A015,MT-011,2026-08-19 17:00:00,vibracao_axial,vibracao,1.1,1.2,-0.1,mm/s,leve,operacao_normal
A016,MT-042,2026-08-20 07:30:00,rolamento_temp,temperatura,48,45,3,°C,leve,manutencao_preventiva
A017,MT-017,2026-08-20 08:50:00,temp_enrolamento,temperatura,85,60,25,°C,critico,anomalia_eletrica
A018,MT-005,2026-08-20 10:10:00,vibracao_radial,vibracao,2.4,1.0,1.4,mm/s,critico,anomalia_mecanica
A019,MT-029,2026-08-20 11:35:00,isolamento,resistencia,1.1,5.0,-3.9,MΩ,critico,anomalia_eletrica
A020,MT-011,2026-08-20 13:00:00,corrente_fase,corrente,11.5,11.0,0.5,A,leve,operacao_normal
"""

ALERTS_REF_SUMMARIES_CSV = '''alert_id,resumo_referencia
A001,"Alerta moderado detectado no Motor MT-042. A temperatura do enrolamento apresentou desvio de +18°C acima do baseline (78°C vs. 60°C). Recomenda-se verificação do sistema de refrigeração."
A002,"Alerta crítico detectado no Motor MT-017. A vibração radial atingiu 2.1 mm/s, um desvio de +1.1 mm/s acima do baseline. Recomenda-se inspeção imediata do alinhamento e dos rolamentos."
A005,"Alerta crítico detectado no Motor MT-029. A vibração axial atingiu 3.4 mm/s, desvio de +2.2 mm/s sobre o baseline. Recomenda-se parada programada para inspeção mecânica."
A006,"Alerta crítico detectado no Motor MT-011. A resistência de isolamento caiu para 0.4 MΩ, desvio de -4.6 MΩ abaixo do baseline. Risco de falha elétrica; recomenda-se inspeção do isolamento imediatamente."
A011,"Alerta crítico detectado no Motor MT-042. A temperatura do enrolamento atingiu 92°C, desvio de +32°C acima do baseline. Risco iminente de dano térmico; recomenda-se parada do equipamento e inspeção elétrica."
A013,"Alerta crítico detectado no Motor MT-005. A temperatura do rolamento atingiu 70°C, desvio de +25°C acima do baseline. Recomenda-se lubrificação e inspeção mecânica do rolamento."
A017,"Alerta crítico detectado no Motor MT-017. A temperatura do enrolamento atingiu 85°C, desvio de +25°C acima do baseline. Recomenda-se verificação do sistema de refrigeração e da carga aplicada."
A018,"Alerta crítico detectado no Motor MT-005. A vibração radial atingiu 2.4 mm/s, desvio de +1.4 mm/s acima do baseline. Recomenda-se inspeção do alinhamento e balanceamento."
A019,"Alerta crítico detectado no Motor MT-029. A resistência de isolamento caiu para 1.1 MΩ, desvio de -3.9 MΩ abaixo do baseline. Recomenda-se inspeção do sistema de isolamento elétrico."
'''

if (DATA_DIR / 'alerts_raw.csv').exists():
    alerts = pd.read_csv(DATA_DIR / 'alerts_raw.csv', parse_dates=['timestamp'])
else:
    alerts = pd.read_csv(StringIO(ALERTS_RAW_CSV), parse_dates=['timestamp'])

alerts.head()

## 1. Geração de Resumos Textuais de Alertas

Três templates de narrativa (leve, moderado, crítico), com vocabulário e tom proporcionais
à urgência. Os campos do alerta (sensor, magnitude, desvio, timestamp) alimentam o template.

In [ ]:
SENSOR_PT = {
    'temp_enrolamento': 'temperatura do enrolamento',
    'vibracao_radial': 'vibração radial',
    'vibracao_axial': 'vibração axial',
    'corrente_fase': 'corrente de fase',
    'isolamento': 'resistência de isolamento',
    'rolamento_temp': 'temperatura do rolamento',
}

RECOMENDACAO = {
    'temp_enrolamento': 'verificação do sistema de refrigeração',
    'vibracao_radial': 'inspeção do alinhamento e balanceamento',
    'vibracao_axial': 'inspeção mecânica do acoplamento e rolamentos',
    'corrente_fase': 'verificação da carga acoplada e do balanceamento de fases',
    'isolamento': 'inspeção do sistema de isolamento elétrico',
    'rolamento_temp': 'lubrificação e inspeção mecânica do rolamento',
}

TEMPLATES = {
    'leve': (
        "Alerta leve registrado no Motor {motor_id}. A {sensor} apresentou desvio de "
        "{sinal}{desvio}{unidade} em relação ao baseline ({valor}{unidade} vs. {baseline}{unidade}), "
        "em {timestamp}. Nenhuma ação imediata é necessária; recomenda-se apenas monitoramento contínuo."
    ),
    'moderado': (
        "Alerta moderado detectado no Motor {motor_id}. A {sensor} apresentou desvio de "
        "{sinal}{desvio}{unidade} acima do baseline ({valor}{unidade} vs. {baseline}{unidade}) "
        "registrado em {timestamp}. Recomenda-se {recomendacao} nas próximas 48 horas."
    ),
    'critico': (
        "Alerta CRÍTICO detectado no Motor {motor_id}. A {sensor} atingiu {valor}{unidade}, um desvio "
        "de {sinal}{desvio}{unidade} em relação ao baseline de {baseline}{unidade}, registrado em "
        "{timestamp}. Risco elevado de falha; recomenda-se {recomendacao} com prioridade máxima, "
        "avaliando parada imediata do equipamento."
    ),
}

def gerar_resumo_alerta(row: pd.Series) -> str:
    nivel = row['nivel']
    template = TEMPLATES[nivel]
    sinal = '+' if row['desvio'] >= 0 else ''
    return template.format(
        motor_id=row['motor_id'],
        sensor=SENSOR_PT.get(row['sensor'], row['sensor']),
        sinal=sinal,
        desvio=row['desvio'],
        unidade=row['unidade'],
        valor=row['valor_medido'],
        baseline=row['baseline'],
        timestamp=row['timestamp'].strftime('%d/%m/%Y %H:%M'),
        recomendacao=RECOMENDACAO.get(row['sensor'], 'inspeção técnica'),
    )

alerts['resumo_gerado'] = alerts.apply(gerar_resumo_alerta, axis=1)
for _, r in alerts.head(5).iterrows():
    print(f"[{r['alert_id']}] {r['resumo_gerado']}\n")

## 2. Classificação Textual de Eventos (Zero-Shot)

Categorias: `manutenção corretiva`, `manutenção preventiva`, `anomalia elétrica`,
`anomalia mecânica`, `operação normal`.

Duas abordagens são demonstradas:
- **Zero-shot com transformer** (`facebook/bart-large-mnli`), aplicado ao texto do resumo gerado.
- **Baseline por regras** (fallback determinístico, útil quando não há GPU/tempo de download),
  usado aqui como *silver label* para permitir o cálculo de F1 mesmo offline.

> Em ambiente com acesso à internet/GPU (Colab), a célula do classificador zero-shot roda de
> fato o modelo `bart-large-mnli`. Caso não haja acesso, o notebook cai automaticamente no
> classificador de regras para manter o pipeline executável ponta a ponta.

In [ ]:
CATEGORIAS = [
    'manutenção corretiva', 'manutenção preventiva',
    'anomalia elétrica', 'anomalia mecânica', 'operação normal',
]

LABEL_MAP = {
    'manutencao_corretiva': 'manutenção corretiva',
    'manutencao_preventiva': 'manutenção preventiva',
    'anomalia_eletrica': 'anomalia elétrica',
    'anomalia_mecanica': 'anomalia mecânica',
    'operacao_normal': 'operação normal',
}
alerts['label_verdadeiro'] = alerts['evento_tipo'].map(LABEL_MAP)

def classificador_regras(row: pd.Series) -> str:
    """Fallback determinístico baseado em sensor + nível, usado sem dependência de rede/GPU."""
    sensor_eletrico = row['sensor'] in ('temp_enrolamento', 'corrente_fase', 'isolamento')
    if row['nivel'] == 'critico':
        return 'manutenção corretiva'
    if row['nivel'] == 'moderado':
        return 'anomalia elétrica' if sensor_eletrico else 'anomalia mecânica'
    if row['nivel'] == 'leve' and row['desvio'] > 0 and row['desvio'] <= 5:
        return 'manutenção preventiva' if not sensor_eletrico else 'operação normal'
    return 'operação normal'

# USAR_ZERO_SHOT=False evita o download/inferência do modelo (útil para testar rápido no Colab
# sem GPU). Deixe True para a classificação real via bart-large-mnli.
USAR_ZERO_SHOT = True

MODO_CLASSIFICACAO = 'regras (fallback offline)'
if USAR_ZERO_SHOT:
    try:
        import torch
        from transformers import pipeline

        device = 0 if torch.cuda.is_available() else -1
        if device == -1:
            print('Aviso: nenhuma GPU detectada. No Colab, ative em Ambiente de execução > '
                  'Alterar tipo de ambiente de execução > GPU (T4) para acelerar bastante esta etapa.')

        zero_shot = pipeline('zero-shot-classification', model='facebook/bart-large-mnli', device=device)

        # Processamento em LOTE (todos os textos de uma vez) em vez de linha a linha: o pipeline
        # do transformers paraleliza internamente, reduzindo bastante o tempo total (~20 alertas
        # x 5 categorias = 100 comparações, mas em lote roda em uma fração do tempo sequencial).
        textos = alerts['resumo_gerado'].tolist()
        saidas = zero_shot(textos, candidate_labels=CATEGORIAS,
                            hypothesis_template='Este evento é um caso de {}.',
                            batch_size=8)
        labels_preditas = [out['labels'][0] for out in saidas]
        alerts['label_predito'] = labels_preditas
        MODO_CLASSIFICACAO = 'zero-shot (bart-large-mnli)'
    except Exception as e:
        print('Zero-shot indisponível/falhou neste ambiente, usando classificador de regras. Detalhe:', e)
        alerts['label_predito'] = alerts.apply(classificador_regras, axis=1)
else:
    alerts['label_predito'] = alerts.apply(classificador_regras, axis=1)

print('Modo de classificação em uso:', MODO_CLASSIFICACAO)
alerts[['alert_id', 'motor_id', 'sensor', 'nivel', 'label_verdadeiro', 'label_predito']].head(10)

In [ ]:
from sklearn.metrics import f1_score, classification_report

report = classification_report(
    alerts['label_verdadeiro'], alerts['label_predito'],
    labels=CATEGORIAS, zero_division=0,
)
print(report)

f1_macro = f1_score(alerts['label_verdadeiro'], alerts['label_predito'], labels=CATEGORIAS, average='macro', zero_division=0)
print(f'F1-macro: {f1_macro:.3f}')

## 3. Relatório Operacional em Linguagem Natural (com rastreabilidade)

O relatório agrega os alertas do período, destaca equipamentos em risco e tendências, e cada
afirmação referencia o `alert_id` / `sensor` de origem para garantir rastreabilidade.

In [ ]:
def gerar_relatorio(df: pd.DataFrame, periodo_label: str) -> str:
    linhas = [f"# Relatório Operacional — {periodo_label}\n"]

    total = len(df)
    criticos = df[df['nivel'] == 'critico']
    moderados = df[df['nivel'] == 'moderado']
    linhas.append(
        f"No período analisado, foram emitidos **{total} alertas**, sendo "
        f"**{len(criticos)} críticos** e **{len(moderados)} moderados** "
        f"(fonte: {', '.join(df['alert_id'])}).\n"
    )

    linhas.append("## Equipamentos em Risco\n")
    risco = df[df['nivel'].isin(['critico', 'moderado'])].groupby('motor_id')
    if len(risco) == 0:
        linhas.append("Nenhum equipamento apresentou desvio moderado ou crítico no período.\n")
    for motor_id, grupo in risco:
        ids = ', '.join(grupo['alert_id'])
        sensores = ', '.join(sorted(set(SENSOR_PT.get(s, s) for s in grupo['sensor'])))
        pior_nivel = 'crítico' if (grupo['nivel'] == 'critico').any() else 'moderado'
        linhas.append(
            f"- **{motor_id}**: nível mais alto = {pior_nivel}; sensores afetados: {sensores} "
            f"(alertas de origem: {ids})."
        )
    linhas.append('')

    linhas.append("## Tendências Observadas\n")
    por_sensor = df.groupby('sensor')['desvio'].mean().sort_values(ascending=False)
    for sensor, media in por_sensor.items():
        ids = ', '.join(df[df['sensor'] == sensor]['alert_id'])
        linhas.append(
            f"- **{SENSOR_PT.get(sensor, sensor)}**: desvio médio de {media:+.1f} no período "
            f"(dados de origem: {ids})."
        )
    linhas.append('')

    linhas.append("## Recomendações Preliminares\n")
    for _, r in criticos.iterrows():
        linhas.append(f"- [{r['alert_id']}] {r['motor_id']}: {RECOMENDACAO.get(r['sensor'], 'inspeção técnica')} com prioridade máxima.")
    for _, r in moderados.iterrows():
        linhas.append(f"- [{r['alert_id']}] {r['motor_id']}: {RECOMENDACAO.get(r['sensor'], 'inspeção técnica')} em até 48h.")

    return '\n'.join(linhas)

relatorio = gerar_relatorio(alerts, 'Semana de 18 a 20/08/2026')
print(relatorio)

try:
    Path('../docs').mkdir(exist_ok=True, parents=True)
    Path('../docs/relatorio_operacional_semanal.md').write_text(relatorio, encoding='utf-8')
except OSError:
    Path('docs').mkdir(exist_ok=True, parents=True)
    Path('docs/relatorio_operacional_semanal.md').write_text(relatorio, encoding='utf-8')

## 4. Avaliação dos Textos Gerados

### 4.1 ROUGE sobre resumos de referência (construídos manualmente)

In [ ]:
try:
    from rouge_score import rouge_scorer
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rouge-score'])
    from rouge_score import rouge_scorer

if (DATA_DIR / 'alerts_reference_summaries.csv').exists():
    ref_df = pd.read_csv(DATA_DIR / 'alerts_reference_summaries.csv')
else:
    ref_df = pd.read_csv(StringIO(ALERTS_REF_SUMMARIES_CSV))

eval_df = ref_df.merge(alerts[['alert_id', 'resumo_gerado']], on='alert_id')

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

resultados = []
for _, r in eval_df.iterrows():
    scores = scorer.score(r['resumo_referencia'], r['resumo_gerado'])
    resultados.append({
        'alert_id': r['alert_id'],
        'rouge1_f': scores['rouge1'].fmeasure,
        'rouge2_f': scores['rouge2'].fmeasure,
        'rougeL_f': scores['rougeL'].fmeasure,
    })

rouge_df = pd.DataFrame(resultados)
display(rouge_df)
print('\nMédias:')
print(rouge_df[['rouge1_f', 'rouge2_f', 'rougeL_f']].mean())

### 4.2 Checklist de Clareza, Precisão e Utilidade

Avaliação manual (checklist) sobre uma amostra de resumos gerados. Critérios: (1) clareza —
linguagem compreensível para operador não especialista; (2) precisão — valores/desvios batem
com os dados de origem; (3) utilidade — inclui recomendação acionável.

In [ ]:
checklist = []
for _, r in alerts.head(10).iterrows():
    checklist.append({
        'alert_id': r['alert_id'],
        'clareza_ok': True,   # linguagem em português técnico acessível, sem jargão excessivo
        'precisao_ok': str(r['desvio']) in r['resumo_gerado'] or str(abs(r['desvio'])) in r['resumo_gerado'],
        'utilidade_ok': ('recomenda' in r['resumo_gerado'].lower()) or (r['nivel'] == 'leve'),
    })
checklist_df = pd.DataFrame(checklist)
checklist_df['aprovado'] = checklist_df[['clareza_ok', 'precisao_ok', 'utilidade_ok']].all(axis=1)
display(checklist_df)
print(f"Taxa de aprovação no checklist: {checklist_df['aprovado'].mean()*100:.1f}%")

In [ ]:
# Persistir outputs para uso no Sprint 4 (contexto operacional injetado no assistente RAG)
try:
    alerts.to_csv('../data/alerts_processed.csv', index=False)
    print('Salvo: ../data/alerts_processed.csv')
except OSError:
    Path('data').mkdir(exist_ok=True, parents=True)
    alerts.to_csv('data/alerts_processed.csv', index=False)
    print('Salvo: data/alerts_processed.csv')